In [ ]:
import os
import gradio as gr
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# ==========================================
# 1. 初始化全域變數與本地大模型
# ==========================================
# 請確保你後台的 Ollama 軟體此時是開啟運行的喔！
chat_model = ChatOllama(model="llama3.2", temperature=0.2)
retriever = None  # 用來儲存上傳 PDF 後建立的臨時檢索器

# ==========================================
# 2. 核心功能函數：處理用戶上傳的 PDF 檔案
# ==========================================
def process_pdf(pdf_file):
    global retriever
    if pdf_file is None:
        return "❌ 請先選擇並上傳 PDF 檔案！"
    
    try:
        status = "📚 正在讀取上傳的 PDF 檔案...\n"
        # pdf_file.name 會自動獲取網頁端上傳後的臨時快取路徑
        loader = PyPDFLoader(pdf_file.name)
        documents = loader.load()
        
        status += "🧬 正在將長文本切分為知識碎片...\n"
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        chunks = text_splitter.split_documents(documents)
        
        status += "💾 正在計算向量並建立本地臨時資料庫 (首次運行可能稍慢)...\n"
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        vector_store = FAISS.from_documents(chunks, embeddings)
        
        # 建立檢索器（每次提問找最相關的 3 個片段）
        retriever = vector_store.as_retriever(search_kwargs={"k": 3})
        
        status += f"✅ 成功！PDF 已成功解析為 {len(chunks)} 個知識片段！現在您可以在右側聊天框提問了。"
        return status
    except Exception as e:
        return f"❌ 解析失敗，錯誤原因: {str(e)}"

# ==========================================
# 3. 核心功能函數：基於 PDF 內容進行對話
# ==========================================
def predict(message, history):
    global retriever
    if retriever is None:
        return "⚠️ 請先在左側上傳 PDF 檔案，並點擊『開始解析 PDF』按鈕！"
    
    # 精心設計的提示詞模板，嚴格限制 AI 不能瞎編
    system_prompt = (
        "你是一個聰明的專業知識庫助手。\n"
        "請仔細閱讀以下已知的内容，並嚴格基於這些内容來回答用戶的問題。\n"
        "如果你在内容中找不到答案，請直接說'抱歉，在您提供的 PDF 中找不到相關答案'，絕對不能瞎編或胡思亂想。\n\n"
        "【已知内容】:\n{context}"
    )
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])
    
    # 組裝 RAG 鏈條
    question_answer_chain = create_stuff_documents_chain(chat_model, prompt)
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)
    
    # 執行檢索與回答
    response = rag_chain.invoke({"input": message})
    return response["answer"]

# ==========================================
# 4. 構建 Gradio 網頁交互介面外觀
# ==========================================
with gr.Blocks(title="本地 PDF 智慧對話機器人") as demo:
    gr.Markdown("# 🤖 本地隱私安全 PDF 智慧對話機器人 (Ollama + Llama 3.2)")
    gr.Markdown("所有的 PDF 解析、向量計算和大模型推理**完全在您本機運行**，0 流量消耗，100% 隱私安全。")
    
    with gr.Row():
        # 左側欄：用來上傳文件和顯示狀態
        with gr.Column(scale=1):
            pdf_input = gr.File(label="第一步：上傳您的 PDF 檔案", file_types=[".pdf"])
            upload_btn = gr.Button("🔥 開始解析 PDF", variant="primary")
            status_output = gr.Textbox(label="系統處理狀態", interactive=False, lines=5)
            
        # 右側欄：聊天視窗
        with gr.Column(scale=2):
            chatbot = gr.ChatInterface(
                fn=predict, 
                title="第二步：針對 PDF 內容進行提問"
            )
            
    # 綁定按鈕點擊事件
    upload_btn.click(fn=process_pdf, inputs=pdf_input, outputs=status_output)

# ==========================================
# 5. 啟動本地網頁服務
# ==========================================
# inbrowser=True 會自動幫你在瀏覽器打開一個新標籤頁
demo.launch(inbrowser=True)